# Phase 1 LayoutLMv3 GPU Fine-tuning

训练数据使用 `local_validation_split_seed_42`，不是 official test。完整训练尚未运行时，不得填写或推测 fine-tuned 指标。


## 1. 统一 GPU Notebook bootstrap

只修改本单元开头的 `RUNTIME` 和四个云端路径。所有安装均使用当前 Notebook Kernel 的 `sys.executable`；Terminal Python 仅用于提示，不作为训练环境。


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

RUNTIME = "modelscope"  # 或 "colab"
if RUNTIME == "modelscope":
    PROJECT_ROOT = Path("/mnt/workspace/ProcureAgent")
    PROCESSED_DIR = PROJECT_ROOT / "data" / "phase1" / "sroie_task3" / "processed"
    IMAGE_ROOT = Path("/mnt/workspace/SROIE/unpacked/sroie/imgs")
    MODEL_DIR = Path("/mnt/workspace/models/layoutlmv3-base")
else:
    PROJECT_ROOT = Path("/content/ProcureAgent")
    PROCESSED_DIR = PROJECT_ROOT / "data" / "phase1" / "sroie_task3" / "processed"
    IMAGE_ROOT = Path("/content/SROIE/unpacked/sroie/imgs")
    MODEL_DIR = Path("/content/models/layoutlmv3-base")

subprocess.run(
    [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "phase1" / "bootstrap_gpu_notebook.py"),
        "--project-root", str(PROJECT_ROOT),
        "--processed-dir", str(PROCESSED_DIR),
        "--image-root", str(IMAGE_ROOT),
        "--model-dir", str(MODEL_DIR),
        "--runtime", RUNTIME,
    ],
    cwd=PROJECT_ROOT,
    check=True,
)

BOOTSTRAP_REPORT = PROJECT_ROOT / "reports" / "phase1" / "gpu_notebook_bootstrap.json"
bootstrap_summary = json.loads(BOOTSTRAP_REPORT.read_text(encoding="utf-8"))
assert bootstrap_summary["training_guard_passed"] is True

import torch
import transformers
from procureguard.extraction.datasets import read_processed_jsonl

MODEL_NAME = "microsoft/layoutlmv3-base"
MAX_LENGTH = 512
BATCH_SIZE = 2
GRAD_ACCUMULATION_STEPS = 4
EPOCHS = 5
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
SEED = 42
BASELINE_MACRO_F1 = float(bootstrap_summary["baseline_macro_f1"])
BASELINE_REPORT_PATH = Path(bootstrap_summary["baseline_report_path"])
MODEL_DIR = Path(bootstrap_summary["model_dir"])
PROCESSED_DIR = Path(bootstrap_summary["processed_dir"])
TRAIN_PATH = PROCESSED_DIR / "train.jsonl"
VALIDATION_PATH = PROCESSED_DIR / "validation.jsonl"
train_samples = read_processed_jsonl(TRAIN_PATH)
validation_samples = read_processed_jsonl(VALIDATION_PATH)
baseline_report = json.loads(BASELINE_REPORT_PATH.read_text(encoding="utf-8"))

print("sys.executable=", sys.executable)
print("python_version=", sys.version)
print("torch_version=", torch.__version__)
print("transformers_version=", transformers.__version__)
print("cuda_available=", torch.cuda.is_available())
print("gpu_name=", torch.cuda.get_device_name(0))
print("training_guard_passed=", bootstrap_summary["training_guard_passed"])


## 2. 环境与训练 guard 摘要


In [ ]:
for key, value in bootstrap_summary.items():
    print(f"{key}={value}")
assert bootstrap_summary["training_guard_passed"] is True


## 3. 样本、OCR token 与 bbox


In [ ]:
print("evaluation_split=local_validation_split_seed_42")
print("train_samples=", len(train_samples))
print("validation_samples=", len(validation_samples))
sample = train_samples[0]
print("sample_id=", sample.sample_id)
for token in sample.tokens[:8]:
    print({"text": token.text, "bbox": token.bbox, "confidence": token.confidence})


## 4. Baseline 与 BIO alignment


In [ ]:
from procureguard.extraction.alignment import align_samples

train_bio_labels, train_alignment = align_samples(train_samples)
validation_bio_labels, validation_alignment = align_samples(validation_samples)
print("baseline_macro_f1=", BASELINE_MACRO_F1)
print("train_alignment=", train_alignment)
print("validation_alignment=", validation_alignment)


## 5. 创建本地 LayoutLMv3Processor


In [ ]:
from procureguard.extraction.layoutlmv3_dataset import create_layoutlmv3_processor

processor = create_layoutlmv3_processor(MODEL_DIR, local_files_only=True)


## 6. 创建 Dataset


In [ ]:
from procureguard.extraction.layoutlmv3_dataset import SROIELayoutLMv3Dataset

train_dataset = SROIELayoutLMv3Dataset(
    train_samples, processor, LABEL2ID, max_length=MAX_LENGTH, allow_missing_images=False
)
validation_dataset = SROIELayoutLMv3Dataset(
    validation_samples, processor, LABEL2ID, max_length=MAX_LENGTH, allow_missing_images=False
)


## 7. 创建 DataLoader


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False)


## 8. 单 batch smoke


In [ ]:
batch = next(iter(train_loader))
labels_non_o_count = int(
    ((batch["labels"] != -100) & (batch["labels"] != LABEL2ID["O"])).sum().item()
)
print("batch_keys=", list(batch))
print({key: tuple(value.shape) for key, value in batch.items()})
print("labels_non_o_count=", labels_non_o_count)

from procureguard.extraction.training import TrainingGuardState, validate_training_guard

guard = TrainingGuardState(
    cuda_available=torch.cuda.is_available(),
    train_samples=len(train_samples),
    validation_samples=len(validation_samples),
    labels_non_o_count=labels_non_o_count,
    baseline_report_exists=BASELINE_REPORT_PATH.exists(),
)
validate_training_guard(guard)
print("training_guard=passed")


## 9. 从本地目录加载 microsoft/layoutlmv3-base


In [ ]:
from transformers import LayoutLMv3ForTokenClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LayoutLMv3ForTokenClassification.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
).to(device)
print("device=", device)


## 10. optimizer


In [ ]:
from torch.optim import AdamW

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)


## 11. scheduler


In [ ]:
from transformers import get_linear_schedule_with_warmup

optimizer_steps_per_epoch = max(
    1, (len(train_loader) + GRAD_ACCUMULATION_STEPS - 1) // GRAD_ACCUMULATION_STEPS
)
total_optimizer_steps = optimizer_steps_per_epoch * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=max(1, int(total_optimizer_steps * 0.1)),
    num_training_steps=total_optimizer_steps,
)


## 12. 手写 PyTorch train loop

如遇 CUDA out of memory，先把 `BATCH_SIZE` 从 2 降到 1，保留 `GRAD_ACCUMULATION_STEPS=4`；重启运行环境释放显存后再从头执行。第一次正式训练不修改模型结构，也不做复杂超参数搜索。


In [ ]:
import time
from torch.nn.utils import clip_grad_norm_

def move_batch(batch):
    return {key: value.to(device) for key, value in batch.items()}

def train_one_epoch(model, dataloader):
    model.train()
    optimizer.zero_grad()
    total_loss = 0.0
    for step, batch in enumerate(dataloader, start=1):
        outputs = model(**move_batch(batch))
        raw_loss = outputs.loss
        (raw_loss / GRAD_ACCUMULATION_STEPS).backward()
        total_loss += float(raw_loss.item())
        should_step = step % GRAD_ACCUMULATION_STEPS == 0 or step == len(dataloader)
        if should_step:
            clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()
    return total_loss / max(len(dataloader), 1)


## 13. validation loop


In [ ]:
from seqeval.metrics import f1_score as seqeval_f1

@torch.no_grad()
def validate_token_level(model, dataloader):
    model.eval()
    total_loss = 0.0
    true_sequences, predicted_sequences = [], []
    for batch in dataloader:
        batch = move_batch(batch)
        outputs = model(**batch)
        total_loss += float(outputs.loss.item())
        predictions = outputs.logits.argmax(dim=-1)
        for predicted_row, label_row in zip(predictions, batch["labels"]):
            true_labels, predicted_labels = [], []
            for predicted_id, label_id in zip(predicted_row.tolist(), label_row.tolist()):
                if label_id == -100:
                    continue
                true_labels.append(ID2LABEL[label_id])
                predicted_labels.append(ID2LABEL[predicted_id])
            true_sequences.append(true_labels)
            predicted_sequences.append(predicted_labels)
    return total_loss / max(len(dataloader), 1), seqeval_f1(true_sequences, predicted_sequences)


## 14. gradient clipping

`clip_grad_norm_` 在每次 optimizer step 前执行，默认 `MAX_GRAD_NORM=1.0`。


In [ ]:
print("max_grad_norm=", MAX_GRAD_NORM)
print("gradient_accumulation_steps=", GRAD_ACCUMULATION_STEPS)


## 15. best checkpoint


In [ ]:
from PIL import Image
from procureguard.extraction.metrics import evaluate_field_f1
from procureguard.extraction.training import (
    EpochLog,
    LayoutLMv3TrainingConfig,
    build_training_report,
    save_loss_curve,
    write_training_outputs,
)

CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints" / "phase1" / "layoutlmv3_best"
REPORT_DIR = PROJECT_ROOT / "reports" / "phase1" / "gpu_training"
config = LayoutLMv3TrainingConfig(
    model_name=MODEL_NAME,
    max_length=MAX_LENGTH,
    batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUMULATION_STEPS,
    epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,
    seed=SEED,
)

def decode_validation_fields(model, samples):
    predictions = []
    model.eval()
    with torch.no_grad():
        for sample in samples:
            image = Image.open(sample.image_path).convert("RGB")
            encoding = processor(
                image,
                [token.text for token in sample.tokens],
                boxes=[list(token.bbox) for token in sample.tokens],
                truncation=True,
                padding="max_length",
                max_length=MAX_LENGTH,
                return_tensors="pt",
            )
            word_ids = encoding.word_ids(batch_index=0)
            tensors = {key: value.to(device) for key, value in encoding.items()}
            predicted_ids = model(**tensors).logits.argmax(dim=-1)[0].tolist()
            word_predictions = {}
            for token_index, word_index in enumerate(word_ids):
                if word_index is not None and word_index not in word_predictions:
                    word_predictions[word_index] = ID2LABEL[predicted_ids[token_index]]
            fields = {"company": "", "address": "", "date": "", "total": ""}
            active_field = None
            for word_index, token in enumerate(sample.tokens):
                label = word_predictions.get(word_index, "O")
                if label == "O":
                    active_field = None
                    continue
                prefix, field_name = label.split("-", 1)
                field_name = field_name.lower()
                if prefix == "B":
                    fields[field_name] = token.text
                    active_field = field_name
                elif prefix == "I" and active_field == field_name:
                    fields[field_name] = (fields[field_name] + " " + token.text).strip()
            predictions.append(fields)
    return predictions

epoch_logs = []
best_field_macro_f1 = -1.0
best_validation_loss = float("inf")
best_epoch = None
for epoch in range(1, EPOCHS + 1):
    epoch_started = time.perf_counter()
    train_loss = train_one_epoch(model, train_loader)
    validation_loss, token_f1 = validate_token_level(model, validation_loader)
    field_predictions = decode_validation_fields(model, validation_samples)
    field_metrics = evaluate_field_f1(
        field_predictions,
        [sample.labels for sample in validation_samples],
    )
    field_macro_f1 = sum(metric.f1 for metric in field_metrics) / len(field_metrics)
    log = EpochLog(
        epoch=epoch,
        train_loss=train_loss,
        validation_loss=validation_loss,
        token_f1=token_f1,
        field_macro_f1=field_macro_f1,
        learning_rate=scheduler.get_last_lr()[0],
        elapsed_time=time.perf_counter() - epoch_started,
    )
    epoch_logs.append(log)
    print(log)
    is_better = (
        field_macro_f1 > best_field_macro_f1
        or (
            field_macro_f1 == best_field_macro_f1
            and validation_loss < best_validation_loss
        )
    )
    if is_better:
        best_field_macro_f1 = field_macro_f1
        best_validation_loss = validation_loss
        best_epoch = epoch
        model.save_pretrained(CHECKPOINT_DIR)
        processor.save_pretrained(CHECKPOINT_DIR)


## 16. loss 曲线


In [ ]:
loss_curve_path = save_loss_curve(epoch_logs, REPORT_DIR / "layoutlmv3_loss_curve.png")
print("loss_curve=", loss_curve_path)


## 17. validation token-level F1


In [ ]:
best_model = LayoutLMv3ForTokenClassification.from_pretrained(CHECKPOINT_DIR).to(device)
best_validation_loss, best_token_f1 = validate_token_level(best_model, validation_loader)
print("best_token_f1=", best_token_f1)


## 18. validation field-level F1


In [ ]:
best_field_predictions = decode_validation_fields(best_model, validation_samples)
best_field_metrics = evaluate_field_f1(
    best_field_predictions,
    [sample.labels for sample in validation_samples],
)
fine_tuned_macro_f1 = sum(metric.f1 for metric in best_field_metrics) / len(best_field_metrics)
for metric in best_field_metrics:
    print(metric)
print("fine_tuned_macro_f1=", fine_tuned_macro_f1)


## 19. baseline vs fine-tuned 对比表


In [ ]:
baseline_by_field = {
    item["field"]: item["f1"] for item in baseline_report["metrics"]
}
print("| field | baseline_f1 | fine_tuned_f1 | improvement |")
print("| --- | ---: | ---: | ---: |")
for metric in best_field_metrics:
    baseline_value = baseline_by_field[metric.field]
    print(
        f"| {metric.field} | {baseline_value:.4f} | {metric.f1:.4f} | "
        f"{metric.f1 - baseline_value:.4f} |"
    )
print(
    f"| macro | {BASELINE_MACRO_F1:.4f} | {fine_tuned_macro_f1:.4f} | "
    f"{fine_tuned_macro_f1 - BASELINE_MACRO_F1:.4f} |"
)


## 20. 错误分析


In [ ]:
from collections import Counter
from procureguard.extraction.error_analysis import collect_error_cases
from procureguard.extraction.alignment import field_target, token_piece

validation_errors = collect_error_cases(
    [sample.sample_id for sample in validation_samples],
    best_field_predictions,
    [sample.labels for sample in validation_samples],
)
unaligned_keys = {
    (case.sample_id, case.field) for case in validation_alignment.unaligned_cases
}
sample_by_id = {sample.sample_id: sample for sample in validation_samples}

def classify_validation_error(error):
    sample = sample_by_id[error.sample_id]
    if (error.sample_id, error.field) in unaligned_keys:
        return "alignment miss"
    if len(sample.tokens) >= MAX_LENGTH - 2:
        return "truncation"
    ocr_text = "".join(token_piece(token, error.field) for token in sample.tokens)
    target = field_target(error.ground_truth or "", error.field)
    if target and target not in ocr_text:
        return "OCR error"
    average_confidence = (
        sum(token.confidence for token in sample.tokens) / len(sample.tokens)
        if sample.tokens else 0.0
    )
    if average_confidence < 0.7:
        return "low quality"
    if error.predicted != error.ground_truth:
        return "model classification miss"
    return "unknown"

categorized_errors = [
    (error, classify_validation_error(error)) for error in validation_errors
]
error_counts = Counter(error.field for error, _ in categorized_errors)
category_counts = Counter(category for _, category in categorized_errors)
error_report_path = PROJECT_ROOT / "reports" / "phase1" / "layoutlmv3_validation_errors.md"
lines = [
    "# LayoutLMv3 Validation Errors",
    "",
    f"- error_count_by_field: {dict(error_counts)}",
    f"- error_type_distribution: {dict(category_counts)}",
    "",
    "| sample_id | field | predicted | ground_truth | category |",
    "| --- | --- | --- | --- | --- |",
]
for error, category in categorized_errors[:10]:
    lines.append(
        f"| {error.sample_id} | {error.field} | {error.predicted or ''} | "
        f"{error.ground_truth or ''} | {category} |"
    )
error_report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
print("error_report=", error_report_path)


## 21. 导出训练报告


In [ ]:
training_report = build_training_report(
    config=config,
    logs=epoch_logs,
    baseline_macro_f1=BASELINE_MACRO_F1,
)
output_paths = write_training_outputs(training_report, REPORT_DIR)
print(output_paths)


## 22. 导出最佳 checkpoint


In [ ]:
import shutil

archive_path = shutil.make_archive(
    str(CHECKPOINT_DIR),
    "zip",
    root_dir=CHECKPOINT_DIR.parent,
    base_dir=CHECKPOINT_DIR.name,
)
print("checkpoint=", CHECKPOINT_DIR)
print("archive=", archive_path)
print("请手动下载 zip，或复制到云端持久化存储。不要提交到 Git。")


## 23. 下一步 API 接入说明


In [ ]:
print(
    "训练完成并经总控审查后，才考虑让上传接口调用 checkpoint 生成 ExtractedFields。"
    "本 Notebook 不修改 API、共享 schema、Agent 主链或 Risk Engine。"
)
